In [1]:
"""
=============================================================================
PARTIE B — Robustesse temporelle : avant et après la crise de 2008
=============================================================================
Tableau comparatif des coefficients CS-ARDL(1,1) pour :
  - Dette des ménages  (debt_hh)
  - Dette des entreprises (debt_corp)
  - Dette publique     (debt_gov)

Sous-périodes :
  - Échantillon complet  (1996–2024)
  - Pré-crise            (1996–2007)
  - Post-crise           (2008–2024)

Pour chaque période × variable : effet court terme (CT), effet long terme (LT),
p-value et étoiles de significativité.

⚠  Mettre à jour DATA_PATH si vous exécutez ce script depuis un autre dossier.
=============================================================================
"""

import warnings
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore")

# ── Chemin vers la base maître ─────────────────────────────────────────────
DATA_PATH = "dette_master.csv"

# ── Variables d'intérêt ────────────────────────────────────────────────────
X_VARS = ["debt_hh", "debt_corp", "debt_gov"]

LABELS = {
    "debt_hh":   "Dette des ménages",
    "debt_corp": "Dette des entreprises",
    "debt_gov":  "Dette publique",
}


# =============================================================================
# 1.  Fonctions utilitaires  (add_lags, add_cs_means, run_cs_ardl, extract)
# =============================================================================

def add_lags(df, group_col, vars_list, max_lag, suffix="lag"):
    """Ajoute les retards de 1 à max_lag pour chaque variable de vars_list."""
    out = df.copy()
    for v in vars_list:
        for L in range(1, max_lag + 1):
            out[f"{v}_{suffix}{L}"] = out.groupby(group_col)[v].shift(L)
    return out


def add_cs_means(df, time_col, vars_list, prefix="cs_"):
    """Ajoute les moyennes transversales (Cross-Sectional means) par année."""
    out = df.copy()
    for c in vars_list:
        out[f"{prefix}{c}"] = out.groupby(time_col)[c].transform("mean")
    return out


def run_cs_ardl(df_input, label="Full", p=1, q=1, x_vars_local=None, verbose=False):
    """
    Estime un modèle CS-ARDL(p, q) sur le sous-échantillon df_input.

    Paramètres
    ----------
    df_input    : DataFrame (country, year, growth, debt_*)
    label       : nom affiché dans les logs
    p           : ordre AR sur y (growth)
    q           : ordre DL sur les dettes
    x_vars_local: liste des variables de dette (défaut : X_VARS global)
    verbose     : affiche le résumé complet PanelOLS si True

    Retourne
    --------
    res  : résultat PanelOLS (ou None si données insuffisantes)
    meta : dict avec n_countries, n_obs, r2_within, phi
    """
    if x_vars_local is None:
        x_vars_local = X_VARS

    df_cs = df_input.copy()

    # Lags sur y et les x
    df_cs = add_lags(df_cs, "country", ["growth"] + x_vars_local,
                     max_lag=max(p, q), suffix="lag")

    # Régresseurs : AR(p) + x_it contemporain + DL(q) sur les dettes
    regressors = [f"growth_lag{L}" for L in range(1, p + 1)]
    regressors += x_vars_local
    regressors += [f"{v}_lag{L}" for v in x_vars_local for L in range(1, q + 1)]

    # CS-means recalculées sur le sous-échantillon (important pour la robustesse)
    df_cs = add_cs_means(df_cs, "year", vars_list=regressors, prefix="cs_")
    cs_regressors = [f"cs_{c}" for c in regressors]

    # Nettoyage
    needed = ["country", "year", "growth"] + regressors + cs_regressors
    df_cs = df_cs[needed].dropna().copy()

    n_entities = df_cs["country"].nunique()
    n_obs = len(df_cs)

    # Seuil minimal de données
    if n_entities < 5 or n_obs < 30:
        print(f"  [SKIP] « {label} » : {n_entities} pays, {n_obs} obs — trop peu de données.")
        return None, {}

    df_cs = df_cs.set_index(["country", "year"])
    rhs = " + ".join(regressors + cs_regressors)
    model = PanelOLS.from_formula(
        f"growth ~ {rhs} + EntityEffects",
        data=df_cs,
        drop_absorbed=True,
    )
    res = model.fit(cov_type="clustered", cluster_entity=True)

    if verbose:
        print(f"\n{'=' * 70}")
        print(f"  CS-ARDL({p},{q}) — {label}")
        print(f"  Pays : {n_entities}  |  Obs : {n_obs}")
        print(f"{'=' * 70}")
        print(res.summary)

    meta = {
        "n_countries": n_entities,
        "n_obs": n_obs,
        "r2_within": res.rsquared_within,
        "phi": float(res.params.get("growth_lag1", np.nan)),
    }
    return res, meta


def extract_effects(res, x_vars_local=None, q=1):
    """
    Calcule les effets de Court Terme (CT) et Long Terme (LT).

    CT(x)  = β₀                            (coeff de x_it)
    LT(x)  = (β₀ + β₁ + … + β_q) / (1 − φ)
    """
    if x_vars_local is None:
        x_vars_local = X_VARS

    nan_dict = {v: np.nan for v in x_vars_local}
    if res is None:
        return nan_dict, nan_dict, nan_dict

    params = res.params
    pvals  = res.pvalues
    phi    = float(params.get("growth_lag1", np.nan))

    ct, lt, pv = {}, {}, {}
    for var in x_vars_local:
        b0 = float(params.get(var, np.nan))
        ct[var] = b0
        pv[var] = float(pvals.get(var, np.nan))

        if np.isnan(b0) or np.isnan(phi) or phi >= 1:
            lt[var] = np.nan
        else:
            b_lags = [float(params.get(f"{var}_lag{L}", 0.0)) for L in range(1, q + 1)]
            lt[var] = (b0 + sum(b_lags)) / (1 - phi)

    return ct, lt, pv


def stars(pval):
    """Étoiles de significativité standard."""
    if np.isnan(pval):
        return ""
    if pval < 0.01:
        return "***"
    if pval < 0.05:
        return "**"
    if pval < 0.10:
        return "*"
    return ""


def fmt_coef(val, pval=np.nan):
    """Formate un coefficient : '−0.3291***' ou '—' si absent."""
    if val is None or np.isnan(val):
        return "—"
    return f"{val:+.4f}{stars(pval)}"


# =============================================================================
# 2.  Chargement des données
# =============================================================================

print("Chargement des données...")
df_master = pd.read_csv(DATA_PATH)
df_master = df_master.sort_values(["country", "year"]).reset_index(drop=True)

print(f"  → {df_master['country'].nunique()} pays | "
      f"{df_master['year'].min()}–{df_master['year'].max()} | "
      f"{len(df_master)} observations\n")


# =============================================================================
# 3.  Découpage temporel
# =============================================================================

df_full = df_master.copy()
df_pre  = df_master[df_master["year"] <= 2007].copy()
df_post = df_master[df_master["year"] >= 2008].copy()

print(f"Échantillon complet  : {df_full['year'].min()}–{df_full['year'].max()} "
      f"| {df_full['country'].nunique()} pays, {len(df_full)} obs")
print(f"Pré-crise  (≤ 2007)  : {df_pre['year'].min()}–{df_pre['year'].max()} "
      f"| {df_pre['country'].nunique()} pays, {len(df_pre)} obs")
print(f"Post-crise (≥ 2008)  : {df_post['year'].min()}–{df_post['year'].max()} "
      f"| {df_post['country'].nunique()} pays, {len(df_post)} obs\n")


# =============================================================================
# 4.  Estimations CS-ARDL(1,1)
# =============================================================================

print("Estimation CS-ARDL(1,1) — échantillon complet...")
res_full, meta_full = run_cs_ardl(df_full,  label="Complet (1996–2024)")

print("Estimation CS-ARDL(1,1) — pré-crise...")
res_pre,  meta_pre  = run_cs_ardl(df_pre,   label="Pré-crise (1996–2007)")

print("Estimation CS-ARDL(1,1) — post-crise...")
res_post, meta_post = run_cs_ardl(df_post,  label="Post-crise (2008–2024)")

print()


# =============================================================================
# 5.  Extraction des effets CT / LT
# =============================================================================

ct_full, lt_full, pv_full = extract_effects(res_full)
ct_pre,  lt_pre,  pv_pre  = extract_effects(res_pre)
ct_post, lt_post, pv_post = extract_effects(res_post)


# =============================================================================
# 6.  Construction du tableau comparatif
# =============================================================================

rows = []
for var in X_VARS:
    rows.append({
        "Type de dette":             LABELS[var],

        # ── Échantillon complet ───────────────────────────────────────
        "CT — Complet":              fmt_coef(ct_full[var], pv_full[var]),
        "LT — Complet":              fmt_coef(lt_full[var]),

        # ── Pré-crise ─────────────────────────────────────────────────
        "CT — Pré-crise (≤ 2007)":  fmt_coef(ct_pre[var],  pv_pre[var]),
        "LT — Pré-crise (≤ 2007)":  fmt_coef(lt_pre[var]),

        # ── Post-crise ────────────────────────────────────────────────
        "CT — Post-crise (≥ 2008)": fmt_coef(ct_post[var], pv_post[var]),
        "LT — Post-crise (≥ 2008)": fmt_coef(lt_post[var]),
    })

df_table = pd.DataFrame(rows).set_index("Type de dette")


# =============================================================================
# 7.  Affichage en console
# =============================================================================

SEP  = "=" * 105
sep2 = "-" * 105

print(SEP)
print("  TABLEAU B — Robustesse temporelle : coefficients CS-ARDL(1,1)")
print("  Impact d'une hausse de 1 point de % du PIB de la dette sur la croissance nominale")
print(SEP)

# En-tête
header = (
    f"{'Type de dette':<26}"
    f"{'CT — Complet':>14}{'LT — Complet':>14}"
    f"{'CT — Pré-crise':>18}{'LT — Pré-crise':>16}"
    f"{'CT — Post-crise':>18}{'LT — Post-crise':>16}"
)
print(header)
print(sep2)

for var in X_VARS:
    row = df_table.loc[LABELS[var]]
    line = (
        f"{LABELS[var]:<26}"
        f"{row['CT — Complet']:>14}{row['LT — Complet']:>14}"
        f"{row['CT — Pré-crise (≤ 2007)']:>18}{row['LT — Pré-crise (≤ 2007)']:>16}"
        f"{row['CT — Post-crise (≥ 2008)']:>18}{row['LT — Post-crise (≥ 2008)']:>16}"
    )
    print(line)

print(sep2)

# Méta-informations
def meta_str(meta):
    if not meta:
        return "N/A"
    return (f"N={meta['n_obs']} obs, {meta['n_countries']} pays, "
            f"R²_within={meta['r2_within']:.3f}, φ={meta['phi']:.3f}")

print(f"\n  Complet   : {meta_str(meta_full)}")
print(f"  Pré-crise : {meta_str(meta_pre)}")
print(f"  Post-crise: {meta_str(meta_post)}")

print(f"""
{SEP}
Notes :
  CT = Effet de Court Terme  = β₀ (coefficient de x_it dans le CS-ARDL)
  LT = Effet de Long Terme   = (β₀ + β₁) / (1 − φ), où φ = coeff. sur growth_lag1
  Significativité  : *** p<0.01  ** p<0.05  * p<0.10   |  — : non estimable
  Erreurs standard : robustes, clusterisées par pays
  CS-means recalculées sur chaque sous-période (correction dépendance cross-sectionnelle)
  ⚠ La période pré-crise ne couvre que ~12 obs/pays → résultats indicatifs
{SEP}
""")


# =============================================================================
# 8.  Export CSV + affichage Pandas stylee (pour Jupyter)
# =============================================================================

csv_path = "tableau_robustesse_temporelle.csv"
df_table.to_csv(csv_path, encoding="utf-8-sig")
print(f"Tableau exporté → {csv_path}")

# ── Affichage enrichi dans Jupyter ──────────────────────────────────────────
try:
    from IPython.display import display

    # On reconstruit un tableau "long" plus lisible en notebook
    rows_display = []
    for var in X_VARS:
        for periode, ct_d, lt_d, pv_d, meta in [
            ("Complet (1996–2024)",   ct_full, lt_full, pv_full, meta_full),
            ("Pré-crise (≤ 2007)",    ct_pre,  lt_pre,  pv_pre,  meta_pre),
            ("Post-crise (≥ 2008)",   ct_post, lt_post, pv_post, meta_post),
        ]:
            rows_display.append({
                "Type de dette": LABELS[var],
                "Période":       periode,
                "Effet CT":      fmt_coef(ct_d[var], pv_d[var]),
                "Effet LT":      fmt_coef(lt_d[var]),
                "N obs":         meta.get("n_obs", "—") if meta else "—",
                "R² within":     f"{meta['r2_within']:.3f}" if meta else "—",
            })

    df_display = pd.DataFrame(rows_display)
    df_display = df_display.set_index(["Type de dette", "Période"])

    def color_coef(val):
        """Colorie en rouge les effets négatifs significatifs, vert les positifs."""
        if isinstance(val, str) and val not in ("—",):
            num_str = val.replace("***", "").replace("**", "").replace("*", "")
            try:
                num = float(num_str)
                if num < 0 and ("*" in val):
                    return "color: #c0392b; font-weight: bold"
                if num > 0 and ("*" in val):
                    return "color: #27ae60; font-weight: bold"
            except ValueError:
                pass
        return ""

    styled = (
        df_display.style
        .set_caption(
            "Tableau B — Robustesse temporelle | CS-ARDL(1,1) "
            "| *** p<0.01 ** p<0.05 * p<0.10"
        )
        .applymap(color_coef, subset=["Effet CT", "Effet LT"])
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "13px"), ("font-weight", "bold"),
                       ("text-align", "left"), ("padding-bottom", "8px")]},
            {"selector": "th",
             "props": [("background-color", "#2c3e50"), ("color", "white"),
                       ("font-size", "12px"), ("text-align", "center")]},
            {"selector": "td",
             "props": [("text-align", "center"), ("font-size", "12px"),
                       ("padding", "5px 10px")]},
            {"selector": "tr:nth-child(even)",
             "props": [("background-color", "#f2f2f2")]},
            {"selector": "tr:hover",
             "props": [("background-color", "#dce9f5")]},
        ])
    )

    display(styled)

except ImportError:
    pass  # pas dans Jupyter, le print console suffit

Chargement des données...
  → 38 pays | 1996–2024 | 1102 observations

Échantillon complet  : 1996–2024 | 38 pays, 1102 obs
Pré-crise  (≤ 2007)  : 1996–2007 | 38 pays, 456 obs
Post-crise (≥ 2008)  : 2008–2024 | 38 pays, 646 obs

Estimation CS-ARDL(1,1) — échantillon complet...
Estimation CS-ARDL(1,1) — pré-crise...
Estimation CS-ARDL(1,1) — post-crise...

  TABLEAU B — Robustesse temporelle : coefficients CS-ARDL(1,1)
  Impact d'une hausse de 1 point de % du PIB de la dette sur la croissance nominale
Type de dette               CT — Complet  LT — Complet    CT — Pré-crise  LT — Pré-crise   CT — Post-crise LT — Post-crise
---------------------------------------------------------------------------------------------------------
Dette des ménages             -0.3288***       -0.0109           +0.0709         +0.0500        -0.5905***         -0.0142
Dette des entreprises            +0.0030       +0.0031         -0.0436**         +0.0340           -0.0063         -0.0009
Dette publique     